# Module 10 Capstone Project Part 4

> Submitted by Phillip Amir Esguerra

In [1]:
%pip install -q langchain langchain_community langchain-pymupdf4llm langchain_huggingface sentence_transformers langchain-chroma langchain_ollama


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Document Ingestion

The Republic Act 12305 or the Philippine National Nuclear Energy Safety Act. I got a downloadable PDF from the senate archives https://legacy.senate.gov.ph/republic_acts/ra%2012305.pdf. This was published September 18, 2025, and we should be at least sure that this does not exist in the current training data of known LLMs.

A text version can be read here: https://lawphil.net/statutes/repacts/ra2025/ra_12305_2025.html which is saved in 'ra 12305-clean.txt'

In [2]:
from langchain_pymupdf4llm import PyMuPDF4LLMLoader

doc_path = r"ra 12305.pdf"
loader = PyMuPDF4LLMLoader(doc_path)
docs = loader.load()

In [3]:
print(f"Loaded {len(docs)} document(s) from '{doc_path}'")

Loaded 50 document(s) from 'ra 12305.pdf'


Create a text splitter of instance `RecursiveCharacterTextSplitter` with chunk size of 1000 and 200 chunk overlap to maintain coherence with nearby chunks.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
doc_chunks = text_splitter.split_documents(docs)

In [5]:
print(f"Chunked {len(doc_chunks)} document(s) from {doc_path}")

Chunked 123 document(s) from ra 12305.pdf


In [6]:
doc_chunks[:3]

[Document(metadata={'producer': '', 'creator': 'Image Capture Plus', 'creationdate': '2025-09-29T01:57:03+00:00', 'source': 'ra 12305.pdf', 'file_path': 'ra 12305.pdf', 'total_pages': 50, 'format': 'PDF 1.5', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-09-29T01:57:03+00:00', 'trapped': '', 'modDate': 'D:20250929015703Z', 'creationDate': 'D:20250929015703Z', 'page': 0}, page_content="S. No. 2899\n\nII, No. 929.1\n\n\nllcpubltc nffljepijtlippinca\n## fflmxgircsgi of pi|tltpjjttt0s\n\n\n( Mctro jillilanila\n\n\n^iucteentlj (llmigress\n\n\n®l[irb ^gular^caaion\n\n\nBegun and held in Metro Manila, on Monday, the twenty-second\nday of July, two thousand twenty-four.\n\n# 1 R epublic A ct N o . 12305]\n\n\nAN ACT PROVIDING FOR A COMPREHENSIVE LEGAL\n\nFRAMEWORK I-'OR NUCLEAR SAFETY. SECURITY.\n\nAND SAFEGUARDS IN THE PEACEFUL\n\nUTILIZATION OF NUCLEAR ENERGY IN THE\n\nPHILIPPINES. ESTABLISHING THE PHILIPPINE\n\nATOMIC ENERGY REGULATORY AUTHORITY. AND\n\nAPPROPRI

## Document Embedding and Vector Store Setup

We'll use [sentence-transformers/all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) as the embedding model and ChromaDB as the vector store.
In `HuggingFaceEmbeddings`, we store the model download cache in the same directory for later cleanup.

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=embedding_model_name, 
    cache_folder="./.cache"
)

/Users/Shared/_ws/nus-gen-ai/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Here, we create an instance of ChromaDB and store its data in a local directory `.chroma_db`. A collection name was also added, but we'll not use it yet.

In [8]:
from langchain_community.vectorstores import Chroma

vector_store = Chroma(
    embedding_function=embeddings,
    persist_directory="./.chroma_db",
    collection_name="ra_12305"
)

ids = vector_store.add_documents(documents=doc_chunks)

/var/folders/5t/yhw_yxtj4hd3cl86y215nn0h0000gn/T/ipykernel_79270/2228445722.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vector_store = Chroma(
/Users/Shared/_ws/nus-gen-ai/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)


In [9]:
results = vector_store.similarity_search(
    "range of civil monetary penalties"
)

print(results[0].metadata)
print("\n---\n")
print(results[0].page_content)

{'source': 'ra 12305.pdf', 'modDate': 'D:20250929015703Z', 'keywords': '', 'moddate': '2025-09-29T01:57:03+00:00', 'subject': '', 'format': 'PDF 1.5', 'producer': '', 'creationDate': 'D:20250929015703Z', 'creator': 'Image Capture Plus', 'page': 45, 'file_path': 'ra 12305.pdf', 'creationdate': '2025-09-29T01:57:03+00:00', 'total_pages': 50, 'trapped': '', 'author': '', 'title': ''}

---

_46_


Administrative penalties imposed by the PhilATOM may
include notice of violation, fine, suspension, modification, cease
and desist order, or revocation of an authorization.


Civil monetary penalties in the form of fines shall be
assessed by the PhilATOM for each violation identified which
shall not be less than Fifty thousand pesos (P50,000) but not
more than One hundred million pesos (P100,000,000);
_l_ _J_ _rouided,_ That in cases of repeated acts of non-compliance or
violation hereof, an additional fine of not less than Ten
thousand pesos (P 10,000) hut not more than One hundred
thousand peso

Visual inspecting the results, it is correct that the results were fetched from page 45 (0-based count).

## Retrieval Augmented Generation

We're going to use a local Ollama setup which we'll load llama3.1 model. The setup will use the default Ollama host at `localhost:11434`

In [10]:
# use the underlying ollama client to list all available models
from ollama import Client as OllamaClient

ollama_client = OllamaClient()
model_list = ollama_client.list()
names = [model.model for model in model_list.models]
print(names)

['embeddinggemma:latest', 'gemma3:270m', 'llama4:latest', 'phi4-reasoning:latest', 'mistral:latest', 'llama3.2:latest', 'e5-large-v2:latest', 'intfloat-e5-large-v2:latest', 'llama3.3:latest', 'llama3.1:8b-instruct-q8_0', 'phi4-mini:latest', 'qwen2.5:3b', 'jeffh/intfloat-multilingual-e5-large:f32', 'nomic-embed-text:latest', 'deepseek-r1:8b-llama-distill-q4_K_M', 'llama3.1:latest', 'deepseek-r1:latest']


In [11]:
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
    temperature=0
)

> According to LangChain docs, `ConversationBufferMemory` is deprecated since version 0.3.1.

In [45]:
from operator import itemgetter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.messages import BaseMessage
from langchain_core.chat_history import InMemoryChatMessageHistory

system_prompt = """You are a legal assistant for question-answering tasks about RA 12305 or the Philippine Nuclear Energy Safety Act.
Use the following retrieved passages to answer the question.
Cite specific sections or articles of the law when relevant.
Use concise, accurate, and formal language appropriate for legal context.
Answer in markdown format.

Context: {context}"""

session_id = "phillip-session"
history = InMemoryChatMessageHistory(session_id=session_id)

# ideally this should get the chat history from another storage i.e. DB
def get_history(session_id: str) -> InMemoryChatMessageHistory:
    return history

retriever = vector_store.as_retriever(search_kwargs={"k": 5})
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("placeholder", "{chat_history}"),
        ("human", "{question}"),
    ]
)
context = itemgetter("question") | retriever
rag_chain = (
    RunnablePassthrough.assign(context=context)
    | prompt
    | llm
)

chain_with_history = RunnableWithMessageHistory(
    rag_chain,
    lambda session_id: history,
    input_messages_key="question",
    history_messages_key="chat_history",
    output_messages_key="answer"
)

config = {"configurable": {"session_id": session_id}}

In [ ]:
from IPython.display import display, Markdown, Latex

question_1 = "What is the role of DENR and PNRI in the national policy and strategy for radioactive waste and spent fuel management?"
result_1 = chain_with_history.invoke(
    {"question": question_1}, 
    config
)

history.add_user_message(question_1)
history.add_ai_message(result_1.content)

display(Markdown(result_1.content))

/Users/Shared/_ws/nus-gen-ai/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Error in RootListenersTracer.on_chain_end callback: KeyError('answer')


According to Section 47(a) of RA 12305, the roles of DENR and PNRI are as follows:

* The DENR shall participate in the development and approval of the national policy and strategy for radioactive waste and spent fuel management.
* The PNRI (Philippine Nuclear Research Institute) shall coordinate with the DENR, PhilATOM, and other relevant agencies to identify a minimum area of two hundred (200) hectares as site for the management of low and intermediate level radioactive waste.

In summary, both DENR and PNRI play important roles in the development and implementation of the national policy and strategy for radioactive waste and spent fuel management.

In [49]:
question_2 = "Will they have an office?"
result_2 = chain_with_history.invoke(
    {"question": question_2}, 
    config
)

history.add_user_message(question_2)
history.add_ai_message(result_2.content)

display(Markdown(result_2.content))

/Users/Shared/_ws/nus-gen-ai/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Error in RootListenersTracer.on_chain_end callback: KeyError('answer')


According to Section 54 of RA 12305, the PhilATOM shall establish its headquarters within the Clark Special Economic Zone in Pampanga and Tarlac, which may be leased from the Bases Conversion and Development Authority (BCDA).

Additionally, Section 54 also states that the PhilATOM "shall establish additional offices in strategic areas as it may deem necessary".

However, there is no specific mention of DENR or PNRI having an office within the PhilATOM headquarters or any other location.

In [52]:
question_3 = "summarize your past two responses"
result_3 = chain_with_history.invoke(
    {"question": question_3}, 
    config
)

history.add_user_message(question_3)
history.add_ai_message(result_3.content)

display(Markdown(result_3.content))

/Users/Shared/_ws/nus-gen-ai/.venv/lib/python3.12/site-packages/torch/nn/modules/module.py:1762: FutureWarning: `encoder_attention_mask` is deprecated and will be removed in version 4.55.0 for `BertSdpaSelfAttention.forward`.
  return forward_call(*args, **kwargs)
Error in RootListenersTracer.on_chain_end callback: KeyError('answer')


Here's a summary:

**Role of DENR and PNRI in national policy and strategy for radioactive waste and spent fuel management:**

* Participate in development and approval of national policy and strategy
* Coordinate with other agencies to identify site for low and intermediate level radioactive waste management

**Office location:**

* PhilATOM will have its headquarters within the Clark Special Economic Zone (Pampanga and Tarlac)
* Additional offices may be established by PhilATOM as needed

In the above Q&A chain, we have to manually add history (I have yet to figure out why but some examples looks like they didn't have to). These Q&A chains asked 2 different questions and at the third instruction, asked the LLM to summarize both responses which shows the capability of the chat history. 

In [58]:
from IPython.display import display, Markdown
from langchain_core.messages import HumanMessage, AIMessage

def format_chat_history(history):
    md_content = "# Chat History\n\n"
    
    for msg in history.messages:
        if hasattr(msg, 'type') and msg.type == 'human':
            md_content += f"**User**: {msg.content}\n\n"
        elif hasattr(msg, 'type') and msg.type == 'ai':
            md_content += f"**AI**: \n\n{msg.content}\n\n"
            md_content += "---\n\n"
    
    return md_content

# Display the chat history in a beautified markdown format
display(Markdown(format_chat_history(history)))

# Chat History

**User**: What is the role of DENR and PNRI in the national policy and strategy for radioactive waste and spent fuel management?

**AI**: 

According to Section 47(a) of RA 12305, a national policy and strategy for radioactive waste and spent fuel management shall be developed and approved at the national level by a presidential issuance, until a law has been promulgated to this effect.

Additionally, Section 47(l) states that the DENR shall, in coordination with the Philippine Nuclear Research Institute (PNRI), the PhilATOM, and other relevant agencies, identify a minimum area of two hundred (200) hectares as site for the management of low and intermediate level radioactive waste which shall be called “The National Low and Intermediate Level Waste Management Site”.

Therefore, the role of DENR and PNRI in the national policy and strategy for radioactive waste and spent fuel management is to:

* Participate in the development and approval of the national policy and strategy
* Coordinate with other relevant agencies in identifying a site for the management of low and intermediate level radioactive waste.

---

**User**: What is the role of DENR and PNRI in the national policy and strategy for radioactive waste and spent fuel management?

**AI**: 

According to Section 47(a) of RA 12305, the roles of DENR and PNRI are as follows:

* The DENR shall participate in the development and approval of the national policy and strategy for radioactive waste and spent fuel management.
* The PNRI (Philippine Nuclear Research Institute) shall coordinate with the DENR, PhilATOM, and other relevant agencies to identify a minimum area of two hundred (200) hectares as site for the management of low and intermediate level radioactive waste.

In summary, both DENR and PNRI play important roles in the development and implementation of the national policy and strategy for radioactive waste and spent fuel management.

---

**User**: Will they have an office?

**AI**: 

According to Section 54 of RA 12305, the PhilATOM shall establish its headquarters within the Clark Special Economic Zone in Pampanga and Tarlac, which may be leased from the Bases Conversion and Development Authority (BCDA).

Additionally, Section 54 also states that the PhilATOM "shall establish additional offices in strategic areas as it may deem necessary".

However, there is no specific mention of DENR or PNRI having an office within the PhilATOM headquarters or any other location.

---

**User**: summarize your past two responses

**AI**: 

Here's a summary:

**Role of DENR and PNRI in national policy and strategy for radioactive waste and spent fuel management:**

* Participate in development and approval of national policy and strategy
* Coordinate with other agencies to identify site for low and intermediate level radioactive waste management

**Office location:**

* PhilATOM will have its headquarters within the Clark Special Economic Zone (Pampanga and Tarlac)
* Additional offices may be established by PhilATOM as needed

---

